# 4. Real data
The estimators from `3_main_experiments.ipynb` on real data: CDC Diabetes Health Indicators (UCI 891,
BRFSS 2015, 253,680 rows, 22 variables) with the expert-elicited DIABETES DAGs from Bayesys
(Zahoor, Constantinou, Curtis & Hasanuzzaman 2025, *Knowledge-Based Systems* 329B:114382), one per
expert confidence level. There is no ground truth here, so **held-out log-likelihood** (mean
per test row of $\sum_i \log q(x_i \mid pa_i)$, higher is better) replaces KL. Logic lives in
`ndbn/real_data.py`; shared speed settings come from `1_mle.ipynb`.

In [1]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

import os
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ndbn.real_data import (load_cdc, discretise, check_marginals, load_bayesys_dag,
                            config_counts, run_real_experiments, node_table)
import ndbn.plotting as plotting
from ndbn.plotting import plot_grouped_curves, plot_node_loglik
from ndbn.notebook_config import load_notebook_config

logging.getLogger("pgmpy").setLevel(logging.ERROR)

# shared settings (VAL_FRAC, N_EPOCHS, CHOSEN_JSON, speed settings) from 1_mle.ipynb
globals().update(load_notebook_config("1_mle.ipynb"))

In [3]:
from IPython.display import HTML, display
display(HTML("""<style>
.cell-output-ipywidget-background { background-color: transparent !important; }
:root {
  --jp-widgets-color: var(--vscode-editor-foreground);
  --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>"""))

## Settings
Discretisation levels (`ndbn/real_data.DISCRETISATIONS`): **L1** reproduces Zahoor et al. Table 1
(mostly binary, Age 6, Income 4, ...); **L2** restores native ordinal cardinality (Age 13, Income 8,
Education 6, GenHlth 5, BMI 4 WHO classes, Ment/PhysHlth 4); **L3** is finer still (BMI deciles,
Ment/PhysHlth 6) and is left out of the default grid. DAG levels are the expert confidence levels.
`N_TRAIN` sizes are nested subsamples of each training fold (`None` = the whole fold, ~203k rows).

In [4]:
# real-data experiment settings (tagged "config")
import os
import torch.nn as nn
from ndbn.experiments import load_chosen_configs

REAL_DIR     = "results6-real"
REAL_CSV     = os.path.join(REAL_DIR, "results.csv")
DAG_PATHS    = {lvl: f"data/bayesys/knowledge_{lvl}_confidence.csv"
                for lvl in ("high", "moderate", "low")}

DISC_LEVELS  = ["L1", "L2"]
DAG_LEVELS   = ["high", "moderate"]
N_TRAIN      = [500, 2000, 10000, 50000, None]   # None = full training fold
K_FOLDS      = 5
NN_MAX_N     = 10000      # NN arms skipped above this (incl. the full fold); tabular arms are not.
                          # Raise to 50000 and re-run the grid: only the missing NN rows are fitted
SEED         = 0
N_JOBS_REAL  = 10         # worker processes for the grid (~1 GB each)

# estimator names must be unique: resume matches on the name
ESTIMATORS = [
    {"name": "mle",           "type": "mle"},                        # diagnostic only
    {"name": "dirichlet_0.5", "type": "dirichlet", "pseudo_count": 0.5},
    {"name": "nn_linear",     "type": "nn", "hidden_dims": (),    "activation": None},
]
# tuned arm(s) chosen at the end of 2_hyperparameter_tuning.ipynb. Until chosen_nn.json exists
# the grid runs without them; re-running the grid later adds only the missing arms.
if os.path.exists(CHOSEN_JSON):
    ESTIMATORS += [{"type": "nn", **c} for c in load_chosen_configs(CHOSEN_JSON)]
else:
    print(f"WARNING: {CHOSEN_JSON} not found, running without the tuned NN arm(s)")


## Data
Fetched once from UCI and cached to `data/cdc_diabetes.csv`. `check_marginals` asserts the L1
discretisation against the published Table 1 percentages (raises on any state more than 1
percentage point off), validating the whole data path before any model is fitted.

In [5]:
raw = load_cdc()
print(raw.shape)

disc, cards = {}, {}
for lvl in ("L1", "L2", "L3"):
    disc[lvl], cards[lvl] = discretise(raw, lvl)
pd.DataFrame(cards).rename_axis("cardinality")

(253680, 22)


,L1,L2,L3
cardinality,,,
Diabetes_binary,2,2,2
HighBP,2,2,2
HighChol,2,2,2
CholCheck,2,2,2
BMI,3,4,10
Smoker,2,2,2
Stroke,2,2,2
HeartDiseaseorAttack,2,2,2
PhysActivity,2,2,2


In [6]:
check_marginals(disc["L1"])
print("L1 marginals match Zahoor et al. Table 1")

L1 marginals match Zahoor et al. Table 1


## DAGs
All Bayesys labels are UCI column names (no renames, no merged or derived variables). The graphs
nest: high ⊆ moderate ⊆ low. The paper reports 37 / 51 / 56 edges; the Bayesys files have
37 / 50 / 55. The high DAG at L1 reproduces the paper's 1,348 free parameters and maximum in-degree 7.
Smoker, Fruits and Veggies have no edges in the high DAG.

In [7]:
dags = {lvl: load_bayesys_dag(p, list(raw.columns)) for lvl, p in DAG_PATHS.items()}

assert {l: G.number_of_edges() for l, G in dags.items()} == {"high": 37, "moderate": 50, "low": 55}
assert set(dags["high"].edges()) <= set(dags["moderate"].edges()) <= set(dags["low"].edges())
high_L1 = config_counts(dags["high"], cards["L1"])
assert high_L1["free_params"].iloc[-1] == 1348 and high_L1["in_degree"].max() == 7

pd.DataFrame({l: {"edges": G.number_of_edges(), "max in-degree": max(d for _, d in G.in_degree()),
                  "isolated": [v for v in G if G.degree(v) == 0]} for l, G in dags.items()})

,high,moderate,low
edges,37,50,55
max in-degree,7,8,12
isolated,"[Smoker, Fruits, Veggies]",[],[]


In [ ]:
import graphviz
from IPython.display import display

# each DAG; red = edges added at that confidence level (not in the level above)
prev_edges = set()
for lvl in ("high", "moderate", "low"):
    G = dags[lvl]
    dot = graphviz.Digraph(
        graph_attr={"rankdir": "TB", "label": f"{lvl} confidence: {G.number_of_edges()} edges",
                    "labelloc": "t", "fontsize": "16"},
        node_attr={"shape": "box", "style": "rounded", "fontsize": "10"},
        edge_attr={"arrowsize": "0.7"},
    )
    for v in G.nodes():
        dot.node(v, label=f"{v}\nin-degree {G.in_degree(v)}")
    for u, v in G.edges():
        dot.edge(u, v, color="red" if prev_edges and (u, v) not in prev_edges else "black")
    display(dot)
    prev_edges = set(G.edges())

### Configuration counts
Per (DAG, discretisation): tabular free parameters grow with the **product** of parent
cardinalities, the NN input with their **sum**. `rows_per_config` = training-fold rows per
possible parent configuration of the largest node (Diabetes_binary in every case).

In [8]:
n_fold = len(raw) * (K_FOLDS - 1) / K_FOLDS
rows = []
for d in ("L1", "L2", "L3"):
    for g in ("high", "moderate", "low"):
        t = config_counts(dags[g], cards[d])
        nodes = t[t["node"] != "TOTAL"]
        rows.append({"disc": d, "dag": g, "free_params": int(t["free_params"].iloc[-1]),
                     "nn_input_total": int(t["nn_input_dim"].iloc[-1]),
                     "max_configs": int(nodes["n_parent_configs"].max()),
                     "rows_per_config": round(n_fold / nodes["n_parent_configs"].max(), 1),
                     "in_grid": d in DISC_LEVELS and g in DAG_LEVELS})
pd.DataFrame(rows)

,disc,dag,free_params,nn_input_total,max_configs,rows_per_config,in_grid
0,L1,high,1348,129,864,234.9,True
1,L1,moderate,4654,162,2592,78.3,True
2,L1,low,95392,176,93312,2.2,False
3,L2,high,7310,214,4160,48.8,True
4,L2,moderate,49496,261,24960,8.1,True
5,L2,low,1622067,281,1597440,0.1,False
6,L3,high,15576,232,10400,19.5,False
7,L3,moderate,93550,285,62400,3.3,False
8,L3,low,9016841,309,8985600,0.0,False


In [9]:
# per node, for the default grid
{(d, g): config_counts(dags[g], cards[d]) for d in DISC_LEVELS for g in DAG_LEVELS}[("L1", "high")]

,node,card,in_degree,n_parent_configs,free_params,nn_input_dim
0,Diabetes_binary,2.0,7,864,864,20
1,MentHlth,3.0,4,72,144,12
2,GenHlth,3.0,3,48,96,12
3,PhysHlth,3.0,3,48,96,12
4,Income,4.0,2,18,54,9
5,HighBP,2.0,3,18,18,8
6,DiffWalk,2.0,2,12,12,8
7,CholCheck,2.0,2,8,8,6
8,NoDocbcCost,2.0,2,8,8,6
9,PhysActivity,2.0,1,6,6,6


## Smoke run
One fold, L1, high-confidence DAG, every N and estimator, serial, so the fit times are clean.
Rows go into the main results file; the full grid below skips them. Check the NN fit times at
the largest N before launching the grid (they set `NN_MAX_N`).

In [10]:
run_real_experiments(
    raw, disc_levels=["L1"], dag_paths={"high": DAG_PATHS["high"]}, estimators=ESTIMATORS,
    n_train=N_TRAIN, k_folds=K_FOLDS, nn_max_n=NN_MAX_N, val_frac=VAL_FRAC, n_epochs=N_EPOCHS,
    results_dir=REAL_DIR, folds_to_run=[0], seed=SEED,
    n_jobs=1, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)
smoke = pd.read_csv(REAL_CSV, dtype={"n_train_spec": str})
smoke = smoke[(smoke.disc_level == "L1") & (smoke.dag_level == "high") & (smoke.fold == 0)]
smoke.pivot_table(index="sample_size", columns="estimator",
                  values=["loglik", "fit_time_s"]).round(3)

(disc, DAG, fold):   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Kaylin\Documents\Neural-Discrete-Bayesian-Network\ndbn\models.py:66: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  idx = torch.as_tensor(data[parents].to_numpy(np.int64), device=DEVICE)


KeyboardInterrupt: 

## Full grid
Every (discretisation, DAG, fold) is one job, run in parallel, fold by fold (fit times from here are inflated by
contention; use the smoke run for timing). Resumable.

**Asymmetry:** NN arms run only up to `NN_MAX_N` training rows; the tabular arms also get the full
training fold (~203k rows). Results at `N = full` therefore compare tabular estimators only.

In [11]:
run_real_experiments(
    raw, disc_levels=DISC_LEVELS, dag_paths={g: DAG_PATHS[g] for g in DAG_LEVELS},
    estimators=ESTIMATORS, n_train=N_TRAIN, k_folds=K_FOLDS, nn_max_n=NN_MAX_N,
    val_frac=VAL_FRAC, n_epochs=N_EPOCHS, results_dir=REAL_DIR, seed=SEED,
    n_jobs=N_JOBS_REAL, device=DEVICE_OVERRIDE, torch_threads=TORCH_THREADS,
)

(disc, DAG, fold):   0%|          | 0/20 [00:00<?, ?it/s]

## Figures
Each point is a mean over folds.

In [ ]:
res = pd.read_csv(REAL_CSV, dtype={"n_train_spec": str})
# only the current estimators (the smoke run also tried others)
res = res[res["estimator"].isin([e["name"] for e in ESTIMATORS])]
plot_dir = os.path.join(REAL_DIR, "plots")
os.makedirs(plot_dir, exist_ok=True)
MIN_OBS = int(res.groupby(['disc_level', 'dag_level'])['fold'].nunique().min())  # folds every panel has

### 1. Held-out log-likelihood vs N
One panel per (discretisation, DAG). Plain MLE is a diagnostic: its zero-probability hits are
floored at `EPS` (see `n_floored`), so its curve is dominated by them at small N: `dirichlet_0.5` is the tabular baseline to compare against.

In [ ]:
fig, axes = plt.subplots(len(DISC_LEVELS), len(DAG_LEVELS), sharex=True,
                         figsize=(6 * len(DAG_LEVELS), 4.5 * len(DISC_LEVELS)), squeeze=False)
for i, d in enumerate(DISC_LEVELS):
    for j, g in enumerate(DAG_LEVELS):
        sub = res[(res.disc_level == d) & (res.dag_level == g)]
        plot_grouped_curves(sub, x="sample_size", hue="estimator", y="loglik", err="sem",
                            min_obs=MIN_OBS, ax=axes[i, j], ylabel="held-out log-lik / row")
        axes[i, j].set_xscale("log")
        axes[i, j].set_title(f"{d}, {g}-confidence DAG")
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "loglik_vs_n.png"), dpi=150)

### 2. Unseen-configuration rate vs discretisation
Fraction of test rows whose parent configuration for at least one node never appeared in training:
the direct measure of where tabular CPTs have no data. The same for every estimator.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, n in zip(axes, (2000, 50000)):
    plot_grouped_curves(res, x="disc_level", hue="dag_level", y="unseen_rate",
                        estimator="dirichlet_0.5", sample_size=n, min_obs=MIN_OBS, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "unseen_rate_vs_disc.png"), dpi=150)

### 3. Free parameters vs discretisation
Tabular parameters grow with the product of parent cardinalities, NN parameters with their sum.

In [ ]:
fig, axes = plt.subplots(1, len(DAG_LEVELS), figsize=(6 * len(DAG_LEVELS), 4.5), squeeze=False)
for ax, g in zip(axes[0], DAG_LEVELS):
    plot_grouped_curves(res[res.dag_level == g], x="disc_level", hue="estimator",
                        y="free_params", sample_size=2000, min_obs=MIN_OBS, logy=True, ax=ax)
    ax.set_title(f"{g}-confidence DAG")
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "free_params_vs_disc.png"), dpi=150)

### 4. Per-node log-likelihood vs parent configurations
Gain of each estimator over `dirichlet_0.5`, per node, against that node's number of parent
configurations. The synthetic result being tested: any NN advantage should be concentrated in the
high in-degree (many-configuration) nodes.

In [ ]:
nodes = node_table(res, DAG_PATHS, raw)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, n in zip(axes, (500, 2000, 10000)):
    plot_node_loglik(nodes[nodes.estimator != "mle"], sample_size=n, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(plot_dir, "node_loglik_vs_configs.png"), dpi=150)